# 🔥 PATTERNSTEIN FUSION - MIMIC-IV Training
## Multi-Modal Attention Fusion for Medical Diagnosis

**Goal:** Train the fusion layer to combine 7 agent embeddings using real MIMIC-IV data

**Dataset:** MIMIC-IV (ICU patients with vitals, labs, notes, diagnoses)

**Architecture:** Multi-head attention fusion → Disease prediction

In [ ]:
# Install required packages
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## Step 1: Load MIMIC-IV Data

**Option A:** If you have MIMIC-IV access:
- Download from PhysioNet
- Use admissions, vitalsigns, labevents, diagnoses_icd

**Option B:** Use synthetic data for proof of concept (we'll create realistic samples)

In [ ]:
# For now, we'll create synthetic multi-modal data that mimics MIMIC-IV structure
# Replace this with real MIMIC-IV data when available

def create_synthetic_multimodal_data(n_patients=5000):
    """
    Create synthetic patient data with 7 modalities:
    1. Vitals (ECG features)
    2. Pathology (tissue markers)
    3. Radiology (imaging features)
    4. Lab Results (blood work)
    5. Genomic (genetic markers)
    6. Movement (activity patterns)
    7. Language (symptom descriptions)
    """
    
    np.random.seed(42)
    
    # Simulate 7 agent embeddings (each agent outputs 64-dim embedding)
    vitals_embeddings = np.random.randn(n_patients, 64)
    pathology_embeddings = np.random.randn(n_patients, 64)
    radiology_embeddings = np.random.randn(n_patients, 64)
    lab_embeddings = np.random.randn(n_patients, 64)
    genomic_embeddings = np.random.randn(n_patients, 64)
    movement_embeddings = np.random.randn(n_patients, 64)
    language_embeddings = np.random.randn(n_patients, 64)
    
    # Stack all embeddings
    all_embeddings = np.stack([
        vitals_embeddings,
        pathology_embeddings,
        radiology_embeddings,
        lab_embeddings,
        genomic_embeddings,
        movement_embeddings,
        language_embeddings
    ], axis=1)  # Shape: (n_patients, 7, 64)
    
    # Create labels (5 disease categories)
    # 0: Healthy, 1: Cardiac, 2: Cancer, 3: Infection, 4: Autoimmune
    labels = np.random.randint(0, 5, size=n_patients)
    
    # Add some correlation between embeddings and labels for realism
    for i in range(n_patients):
        if labels[i] == 1:  # Cardiac
            all_embeddings[i, 0] += 2  # Vitals agent stronger
        elif labels[i] == 2:  # Cancer
            all_embeddings[i, 1] += 2  # Pathology agent stronger
        elif labels[i] == 3:  # Infection
            all_embeddings[i, 3] += 2  # Lab agent stronger
    
    return all_embeddings, labels

# Generate data
X, y = create_synthetic_multimodal_data(n_patients=5000)

print(f"Data shape: {X.shape}")  # (5000, 7, 64)
print(f"Labels shape: {y.shape}")  # (5000,)
print(f"\nLabel distribution:")
print(pd.Series(y).value_counts().sort_index())

## Step 2: Build Multi-Head Attention Fusion Model

In [ ]:
def build_patternstein_fusion_model(n_agents=7, embedding_dim=64, n_classes=5):
    """
    Build the Patternstein Fusion model with multi-head attention
    
    Architecture:
    1. Input: 7 agent embeddings (7, 64)
    2. Multi-head attention layer (learns to weight agents)
    3. Global average pooling
    4. Dense layers for classification
    5. Output: Disease prediction (5 classes)
    """
    
    # Input: (batch_size, 7, 64)
    inputs = keras.Input(shape=(n_agents, embedding_dim), name='agent_embeddings')
    
    # Multi-head attention (the fusion magic!)
    attention_output = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=embedding_dim,
        name='fusion_attention'
    )(inputs, inputs)
    
    # Add & Norm
    x = layers.Add()([inputs, attention_output])
    x = layers.LayerNormalization()(x)
    
    # Feed-forward network
    ff = layers.Dense(256, activation='relu')(x)
    ff = layers.Dropout(0.3)(ff)
    ff = layers.Dense(embedding_dim)(ff)
    
    # Add & Norm again
    x = layers.Add()([x, ff])
    x = layers.LayerNormalization()(x)
    
    # Global pooling to combine all agents
    x = layers.GlobalAveragePooling1D()(x)
    
    # Classification head
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output
    outputs = layers.Dense(n_classes, activation='softmax', name='disease_prediction')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs, name='patternstein_fusion')
    
    return model

# Build model
model = build_patternstein_fusion_model()
model.summary()

## Step 3: Prepare Data & Train

In [ ]:
# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Training set: {X_train.shape[0]} patients")
print(f"Validation set: {X_val.shape[0]} patients")
print(f"Test set: {X_test.shape[0]} patients")

In [ ]:
# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    keras.callbacks.ModelCheckpoint('patternstein_fusion_best.h5', save_best_only=True)
]

# Train!
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## Step 4: Evaluate & Visualize

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\n" + "="*50)
print("🔥 PATTERNSTEIN FUSION RESULTS 🔥")
print("="*50)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")
print("="*50)

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Healthy', 'Cardiac', 'Cancer', 'Infection', 'Autoimmune'],
            yticklabels=['Healthy', 'Cardiac', 'Cancer', 'Infection', 'Autoimmune'])
plt.title('Confusion Matrix - Patternstein Fusion', fontsize=16, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, 
                          target_names=['Healthy', 'Cardiac', 'Cancer', 'Infection', 'Autoimmune']))

## Step 5: Visualize Attention Weights

In [ ]:
# Extract attention weights for visualization
attention_layer = model.get_layer('fusion_attention')

# Create a model that outputs attention weights
attention_model = keras.Model(
    inputs=model.input,
    outputs=[model.output, attention_layer.output]
)

# Get predictions and attention for a sample
sample_idx = 0
sample = X_test[sample_idx:sample_idx+1]
predictions, attention_output = attention_model.predict(sample)

# Visualize
agent_names = ['Vitals', 'Pathology', 'Radiology', 'Lab', 'Genomic', 'Movement', 'Language']

plt.figure(figsize=(12, 6))
attention_weights = np.mean(attention_output[0], axis=0)  # Average across embedding dim
plt.bar(agent_names, attention_weights)
plt.title('Agent Attention Weights - Sample Patient', fontsize=14, fontweight='bold')
plt.xlabel('Agent')
plt.ylabel('Attention Weight')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nPredicted class: {np.argmax(predictions[0])}")
print(f"True class: {y_test[sample_idx]}")

## Step 6: Save Model

In [ ]:
# Save final model
model.save('patternstein_fusion_final.h5')
print("✅ Model saved as 'patternstein_fusion_final.h5'")

# Save for deployment
model.save('patternstein_fusion.h5')
print("✅ Model saved as 'patternstein_fusion.h5' for deployment")

print("\n🔥 TRAINING COMPLETE! 🔥")
print(f"Final Test Accuracy: {test_accuracy*100:.2f}%")

## Next Steps:

1. **Download the model**: `patternstein_fusion.h5`
2. **Deploy to GCP**: Upload to Cloud Storage, update API
3. **Update website**: Connect to real model endpoint
4. **Test with real data**: Use actual MIMIC-IV when available

**For real MIMIC-IV data:**
- Get access at https://physionet.org/
- Replace synthetic data generation with actual patient records
- Align vitals, labs, notes, diagnoses by patient ID
- Extract embeddings from your 7 trained agents
- Retrain fusion layer on real embeddings